# MinIO напрямую: что лежит под Iceberg-таблицей

MinIO говорит на S3 API, так что любой S3-клиент (boto3, aws cli, s3fs) работает
с ним из коробки. Полезно заглянуть «под капот» — увидеть, из каких файлов на самом
деле состоит Iceberg-таблица.

Консоль с теми же данными: http://localhost:9001 (minioadmin/minioadmin).

In [ ]:
!pip install -q boto3 pandas pyarrow

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
)
print("бакеты:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

## Анатомия таблицы: data/ и metadata/

У каждой Iceberg-таблицы два каталога:
- `data/` — parquet-файлы с собственно данными;
- `metadata/` — `*.metadata.json` (версии таблицы), манифесты (avro) и снапшоты.

In [ ]:
resp = s3.list_objects_v2(Bucket="warehouse", Prefix="analytics.db/sales_by_region_product/")
for o in resp.get("Contents", []):
    print(f"{o['Size']:>8}  {o['Key']}")

## Прочитать metadata.json — корень всей таблицы

Файл с максимальным номером — текущая версия. В нём всё: схема, партиционирование,
список снапшотов и указатель на текущий.

In [ ]:
import json

metas = sorted(o["Key"] for o in resp["Contents"] if o["Key"].endswith(".metadata.json"))
meta = json.loads(s3.get_object(Bucket="warehouse", Key=metas[-1])["Body"].read())
print("формат:          ", meta["format-version"])
print("текущий снапшот: ", meta["current-snapshot-id"])
print("всего снапшотов: ", len(meta["snapshots"]))
print("поля:            ", [f["name"] for f in meta["schemas"][-1]["fields"]])

## Прочитать parquet напрямую

Можно — но с оговоркой: **набор файлов на S3 ≠ текущее состояние таблицы**.
В листинге выше наверняка есть файлы старых снапшотов и delete-файлы — сырое
чтение по маске их не различает. Для корректного состояния ходи через каталог
(pyiceberg — ноутбук 04, Spark, `icebergS3()`), а напрямую — для отладки.

In [ ]:
import io
import pandas as pd

pq = next(o["Key"] for o in resp["Contents"] if o["Key"].endswith(".parquet"))
df = pd.read_parquet(io.BytesIO(s3.get_object(Bucket="warehouse", Key=pq)["Body"].read()))
print(pq)
df.head()

## Presigned URL — поделиться файлом без ключей

Ссылка с подписью внутри, живёт указанное время. Стандартный способ отдать файл
наружу, не раздавая креды.

In [ ]:
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": "warehouse", "Key": pq},
    ExpiresIn=300,
)
print(url)

## Итог

S3 — это просто файлы; таблицей их делает каталог (Hive Metastore) и метаданные
Iceberg. Поэтому одна и та же куча parquet'ов читается из Spark, Impala, ClickHouse,
pyiceberg — все смотрят в один metadata.json.